<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 2: Identificación y taxonomía de datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

> **Idea central:** antes de transformar una variable, identifica qué representa, qué operaciones tienen sentido y qué información puede perderse al codificarla.

**En este archivo de jupyter vamos a tratar los siguientes puntos:**

1. Distinguir variables cualitativas, cuantitativas, binarias, nominales y ordinales.
2. Reconocer la diferencia entre el tipo almacenado (`dtype`) y la escala de medición.
3. Diagnosticar variables en un conjunto de datos real con `pandas`.
4. Identificar valores faltantes y elegir una estrategia con criterio.

Al final, usa la sección **Actividad de cierre** para comprobar tus decisiones.


## ¿Qué es un dato y por qué importa su taxonomía?

Un **dato** es una representación de una característica de una entidad, observación o evento. En un dataframe, cada columna representa una variable y cada fila una observación. La taxonomía ayuda a responder preguntas prácticas:

- ¿Tiene sentido calcular un promedio?
- ¿Existe un orden entre las categorías?
- ¿Qué codificación necesita el modelo?
- ¿Qué valores representan ausencia, desconocido o una categoría real?

### Dos perspectivas que conviene separar

| Perspectiva | Pregunta | Ejemplo |
|---|---|---|
| **Tipo almacenado** | ¿Cómo está guardado en Python? | `int64`, `float64`, `object`, `bool` |
| **Significado estadístico** | ¿Qué operaciones representan la realidad? | nominal, ordinal, discreta, continua |

Una columna guardada como número no necesariamente es cuantitativa. Por ejemplo, `1 = rojo`, `2 = azul` sigue siendo **nominal**: los números son etiquetas y no cantidades.

### Escalas y tipos de variable

| Tipo | Característica | Ejemplos | Operaciones razonables |
|---|---|---|---|
| **Binaria** | Dos estados | `yes/no`, `0/1`, presencia/ausencia | proporción, conteo, tasa |
| **Nominal** | Categorías sin orden | trabajo, país, color | frecuencia, moda |
| **Ordinal** | Categorías con orden, sin distancias necesariamente iguales | primaria/secundaria/terciaria, bajo/medio/alto | comparación de orden, mediana |
| **Cuantitativa discreta** | Conteos enteros | número de contactos, hijos, compras | suma, promedio, dispersión |
| **Cuantitativa continua** | Mediciones en una escala | peso, duración, temperatura | operaciones aritméticas y dispersión |

> **Advertencia:** que una variable tenga distribución normal no es un requisito para llamarla cuantitativa. La distribución se estudia después; no define por sí sola el tipo de variable.

Referencias: [pandas `select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html) · [pandas `CategoricalDtype`](https://pandas.pydata.org/docs/user_guide/categorical.html) · [scikit-learn: encoding categorical features](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)

##  Caso de estudio: campañas de marketing bancario

Trabajaremos con `bank.csv`. Cada fila representa un contacto de una campaña telefónica y `y` indica si la persona contrató el producto ofrecido.

Antes de ejecutar el código, predice la clasificación de estas variables:

- `age`, `balance`, `duration`
- `job`, `marital`, `education`, `month`
- `default`, `housing`, `loan`, `y`

La predicción importa: la clasificación que hagas guiará la exploración y la transformación posterior.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
df_bank = pd.read_csv('Data/bank.csv')

In [3]:
df_bank.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [4]:
df_bank.tail()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no
4520,44,entrepreneur,single,tertiary,no,1136,yes,yes,cellular,3,apr,345,2,249,7,other,no


In [5]:
df_bank.shape

(4521, 17)

In [6]:
df_bank.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [7]:
df_bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


### Lectura rápida del esquema

`info()` permite revisar tamaño, tipos almacenados y valores no nulos. Esta salida es un diagnóstico inicial, no una clasificación completa: una variable `object` puede contener categorías, texto libre o fechas mal interpretadas.

En la siguiente tabla verificaremos qué tipo de dato observa `pandas` en cada columna.

In [8]:
schema_bank = pd.DataFrame({
    'tipo_python': df_bank.dtypes.astype(str),
    'valores_unicos': df_bank.nunique(),
    'faltantes': df_bank.isna().sum()
})
schema_bank

,tipo_python,valores_unicos,faltantes
age,int64,67,0
job,object,12,0
marital,object,3,0
education,object,4,0
default,object,2,0
balance,int64,2353,0
housing,object,2,0
loan,object,2,0
contact,object,3,0
day,int64,31,0


## Variables cualitativas y cuantitativas

`select_dtypes()` clasifica por el tipo almacenado, lo que es útil como primer filtro. Después debemos revisar el significado de cada columna y documentar excepciones.

In [9]:
df_bank.head(2)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no


In [10]:
# Obteniendo datos cuantitativos
df_bank_cuantitativos = df_bank.select_dtypes(include='number')
df_bank_cuantitativos.head()

,age,balance,day,duration,campaign,pdays,previous
0,30,1787,19,79,1,-1,0
1,33,4789,11,220,1,339,4
2,35,1350,16,185,1,330,1
3,30,1476,3,199,4,-1,0
4,59,0,5,226,1,-1,0


In [11]:
df_bank_cuantitativos.columns

Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')

#### Cuantitativas discretas y continuas

La separación `number` / `object` no basta. En este dataset, `age`, `balance` y `duration` son medidas cuantitativas; `campaign`, `pdays` y `previous` son conteos o códigos numéricos cuyo significado merece una revisión adicional.

Observa sus rangos y valores únicos antes de elegir una transformación:

In [12]:
columnas_revisar = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
df_bank[columnas_revisar].agg(['min', 'max', 'nunique']).T

,min,max,nunique
age,19,87,67
balance,-3313,71188,2353
duration,4,3025,875
campaign,1,50,32
pdays,-1,871,292
previous,0,25,24


In [13]:
#Obteniendo datos cualitativos
df_bank_categoricos = df_bank.select_dtypes(include='object')
df_bank_categoricos.head()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,unemployed,married,primary,no,no,no,cellular,oct,unknown,no
1,services,married,secondary,no,yes,yes,cellular,may,failure,no
2,management,single,tertiary,no,yes,no,cellular,apr,failure,no
3,management,married,tertiary,no,yes,yes,unknown,jun,unknown,no
4,blue-collar,married,secondary,no,yes,no,unknown,may,unknown,no


In [14]:
df_bank_categoricos.columns

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')

### Variables binarias

Las columnas `default`, `housing`, `loan` y `y` tienen dos categorías. La codificación `yes/no` conserva legibilidad; si un algoritmo necesita números, puede mapearse explícitamente. No conviene asumir que todas las columnas binarias son objetivos: `y` es la variable respuesta en este caso y las otras son predictoras.

In [15]:
#Otra alternativa para obtener datos categóricos
df_bank.select_dtypes(exclude='number')

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,unemployed,married,primary,no,no,no,cellular,oct,unknown,no
1,services,married,secondary,no,yes,yes,cellular,may,failure,no
2,management,single,tertiary,no,yes,no,cellular,apr,failure,no
3,management,married,tertiary,no,yes,yes,unknown,jun,unknown,no
4,blue-collar,married,secondary,no,yes,no,unknown,may,unknown,no
...,...,...,...,...,...,...,...,...,...,...
4516,services,married,secondary,no,yes,no,cellular,jul,unknown,no
4517,self-employed,married,tertiary,yes,yes,yes,unknown,may,unknown,no
4518,technician,married,secondary,no,no,no,cellular,aug,unknown,no
4519,blue-collar,married,secondary,no,no,no,cellular,feb,other,no


In [16]:
df_bank_categoricos['education'].nunique(), df_bank_categoricos['education'].unique()

(4, array(['primary', 'secondary', 'tertiary', 'unknown'], dtype=object))

In [17]:
df_bank_categoricos['education'].value_counts()

education
secondary    2306
tertiary     1350
primary       678
unknown       187
Name: count, dtype: int64

In [18]:
#Guardamos las columnas originales de df_bank
columns_bank = df_bank.columns
columns_bank

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [19]:
#Guardar las columnas de las variables cuantitivas
columns_cuantitativas = df_bank_cuantitativos.columns
columns_cuantitativas

Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')

In [20]:
#Guardar las columnas de las variables categóricas
columns_categoricas = df_bank_categoricos.columns
columns_categoricas

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')

In [21]:
# Categóricos (Multiestado o cualitativos) --->  (nominales, ordinales)
df_bank_categoricos.head()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,unemployed,married,primary,no,no,no,cellular,oct,unknown,no
1,services,married,secondary,no,yes,yes,cellular,may,failure,no
2,management,single,tertiary,no,yes,no,cellular,apr,failure,no
3,management,married,tertiary,no,yes,yes,unknown,jun,unknown,no
4,blue-collar,married,secondary,no,yes,no,unknown,may,unknown,no


In [22]:
# Valores únicos y categorías de la columna default
df_bank_categoricos['default'].unique(), df_bank_categoricos['default'].nunique()

(array(['no', 'yes'], dtype=object), 2)

In [23]:
# Lista de columnas y sus categorías
for col in columns_categoricas:
    print(f'* La columna {col} tiene {df_bank_categoricos[col].nunique()} categorías y son:\n {df_bank_categoricos[col].unique()}')

* La columna job tiene 12 categorías y son:
 ['unemployed' 'services' 'management' 'blue-collar' 'self-employed'
 'technician' 'entrepreneur' 'admin.' 'student' 'housemaid' 'retired'
 'unknown']
* La columna marital tiene 3 categorías y son:
 ['married' 'single' 'divorced']
* La columna education tiene 4 categorías y son:
 ['primary' 'secondary' 'tertiary' 'unknown']
* La columna default tiene 2 categorías y son:
 ['no' 'yes']
* La columna housing tiene 2 categorías y son:
 ['no' 'yes']
* La columna loan tiene 2 categorías y son:
 ['no' 'yes']
* La columna contact tiene 3 categorías y son:
 ['cellular' 'unknown' 'telephone']
* La columna month tiene 12 categorías y son:
 ['oct' 'may' 'apr' 'jun' 'feb' 'aug' 'jan' 'jul' 'nov' 'sep' 'mar' 'dec']
* La columna poutcome tiene 4 categorías y son:
 ['unknown' 'failure' 'other' 'success']
* La columna y tiene 2 categorías y son:
 ['no' 'yes']


### Variables ordinales: el orden debe declararse

`education` representa niveles educativos con un orden razonable. Aun así, la distancia entre niveles no tiene por qué ser igual: pasar de `primary` a `secondary` no equivale necesariamente a pasar de `secondary` a `tertiary`.

`month` tiene una secuencia temporal, pero es una variable cíclica: diciembre y enero están próximos en el calendario. Para un modelo suele ser mejor usar variables seno/coseno o una codificación categórica que imponer una distancia lineal.

In [24]:
# `education` tiene un orden natural; `month` es temporal y conviene tratarlo aparte.
ordinales_cat = ['education']
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna != 'y'
]
binarias_cat = ['default', 'housing', 'loan', 'y']


In [25]:
binarias_cat

['default', 'housing', 'loan', 'y']

In [26]:
# dataframes con datos ordinales categóricos
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat]
df_bank_categoricos_ord.head()

,education
0,primary
1,secondary
2,tertiary
3,tertiary
4,secondary


In [27]:
# Declarar el orden evita que una transformación posterior lo invente o lo pierda.
education_order = ['primary', 'secondary', 'tertiary']
education_dtype = pd.api.types.CategoricalDtype(
    categories=education_order,
    ordered=True
)
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat].copy()
df_bank_categoricos_ord['education'] = df_bank_categoricos_ord['education'].astype(education_dtype)
df_bank_categoricos_ord['education'].dtype

CategoricalDtype(categories=['primary', 'secondary', 'tertiary'], ordered=True, categories_dtype=object)

### Categóricas Nominales

In [28]:
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna not in binarias_cat
]
nominales_cat

['job', 'marital', 'contact', 'month', 'poutcome']

In [29]:
nominales_cat

['job', 'marital', 'contact', 'month', 'poutcome']

In [30]:
# Base de datos con datos nominales categóricos
df_bank_categoricos_nom = df_bank_categoricos[nominales_cat]
df_bank_categoricos_nom


,job,marital,contact,month,poutcome
0,unemployed,married,cellular,oct,unknown
1,services,married,cellular,may,failure
2,management,single,cellular,apr,failure
3,management,married,unknown,jun,unknown
4,blue-collar,married,unknown,may,unknown
...,...,...,...,...,...
4516,services,married,cellular,jul,unknown
4517,self-employed,married,unknown,may,unknown
4518,technician,married,cellular,aug,unknown
4519,blue-collar,married,cellular,feb,other


## Valores faltantes: identificar antes de imputar

Un valor faltante no siempre significa lo mismo: puede ser una medición no realizada, una respuesta omitida o un valor que no aplica. Antes de decidir, cuantifica el problema y consulta el significado de la variable.

En este cuaderno solo hacemos una introducción. La comparación detallada de métodos de imputación continúa en el módulo de **Tratamiento de datos faltantes**.

In [31]:
#Cargamos un dataset de peliculas
df_movie = pd.read_csv('Data/movie_metadata.csv')
df_movie.head()

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,actor_1_name,movie_title,num_voted_users,cast_total_facebook_likes,actor_3_name,facenumber_in_poster,plot_keywords,movie_imdb_link,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.00,178.00,0.00,855.00,Joel David Moore,"1,000.00","760,505,847.00",Action|Adventure|Fantasy|Sci-Fi,CCH Pounder,Avatar,886204,4834,Wes Studi,0.00,avatar|future|marine|native|paraplegic,http://www.imdb.com/title/tt0499549/?ref_=fn_t...,"3,054.00",English,USA,PG-13,"237,000,000.00","2,009.00",936.00,7.90,1.78,33000
1,Color,Gore Verbinski,302.00,169.00,563.00,"1,000.00",Orlando Bloom,"40,000.00","309,404,152.00",Action|Adventure|Fantasy,Johnny Depp,Pirates of the Caribbean: At World's End,471220,48350,Jack Davenport,0.00,goddess|marriage ceremony|marriage proposal|pi...,http://www.imdb.com/title/tt0449088/?ref_=fn_t...,"1,238.00",English,USA,PG-13,"300,000,000.00","2,007.00","5,000.00",7.10,2.35,0
2,Color,Sam Mendes,602.00,148.00,0.00,161.00,Rory Kinnear,"11,000.00","200,074,175.00",Action|Adventure|Thriller,Christoph Waltz,Spectre,275868,11700,Stephanie Sigman,1.00,bomb|espionage|sequel|spy|terrorist,http://www.imdb.com/title/tt2379713/?ref_=fn_t...,994.00,English,UK,PG-13,"245,000,000.00","2,015.00",393.00,6.80,2.35,85000
3,Color,Christopher Nolan,813.00,164.00,"22,000.00","23,000.00",Christian Bale,"27,000.00","448,130,642.00",Action|Thriller,Tom Hardy,The Dark Knight Rises,1144337,106759,Joseph Gordon-Levitt,0.00,deception|imprisonment|lawlessness|police offi...,http://www.imdb.com/title/tt1345836/?ref_=fn_t...,"2,701.00",English,USA,PG-13,"250,000,000.00","2,012.00","23,000.00",8.50,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.00,NaN,Rob Walker,131.00,NaN,Documentary,Doug Walker,Star Wars: Episode VII - The Force Awakens ...,8,143,NaN,0.00,NaN,http://www.imdb.com/title/tt5289954/?ref_=fn_t...,NaN,NaN,NaN,NaN,NaN,NaN,12.00,7.10,NaN,0


In [32]:
#Obtenemos la información general del dataframe
df_movie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      5024 non-null   object 
 1   director_name              4939 non-null   object 
 2   num_critic_for_reviews     4993 non-null   float64
 3   duration                   5028 non-null   float64
 4   director_facebook_likes    4939 non-null   float64
 5   actor_3_facebook_likes     5020 non-null   float64
 6   actor_2_name               5030 non-null   object 
 7   actor_1_facebook_likes     5036 non-null   float64
 8   gross                      4159 non-null   float64
 9   genres                     5043 non-null   object 
 10  actor_1_name               5036 non-null   object 
 11  movie_title                5043 non-null   object 
 12  num_voted_users            5043 non-null   int64  
 13  cast_total_facebook_likes  5043 non-null   int64

In [34]:
# missing_summary
missing_summary = df_movie.isna().agg(['sum', 'mean']).T.rename(columns = {'sum':'n_faltantes', 'mean':'proporcion_faltantes'})
missing_summary

,n_faltantes,proporcion_faltantes
color,19.00,0.00
director_name,104.00,0.02
num_critic_for_reviews,50.00,0.01
duration,15.00,0.00
director_facebook_likes,104.00,0.02
actor_3_facebook_likes,23.00,0.00
actor_2_name,13.00,0.00
actor_1_facebook_likes,7.00,0.00
gross,884.00,0.18
genres,0.00,0.00


In [36]:
df_movie['color']

0       Color
1       Color
2       Color
3       Color
4         NaN
        ...  
5038    Color
5039    Color
5040    Color
5041    Color
5042    Color
Name: color, Length: 5043, dtype: object

### Manejando datos Faltantes (intro)

In [38]:
# Opción 1: eliminar filas completas solo cuando la pérdida sea aceptable.
df_movie_clean = df_movie.dropna()
df_movie_clean

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,actor_1_name,movie_title,num_voted_users,cast_total_facebook_likes,actor_3_name,facenumber_in_poster,plot_keywords,movie_imdb_link,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.00,178.00,0.00,855.00,Joel David Moore,"1,000.00","760,505,847.00",Action|Adventure|Fantasy|Sci-Fi,CCH Pounder,Avatar,886204,4834,Wes Studi,0.00,avatar|future|marine|native|paraplegic,http://www.imdb.com/title/tt0499549/?ref_=fn_t...,"3,054.00",English,USA,PG-13,"237,000,000.00","2,009.00",936.00,7.90,1.78,33000
1,Color,Gore Verbinski,302.00,169.00,563.00,"1,000.00",Orlando Bloom,"40,000.00","309,404,152.00",Action|Adventure|Fantasy,Johnny Depp,Pirates of the Caribbean: At World's End,471220,48350,Jack Davenport,0.00,goddess|marriage ceremony|marriage proposal|pi...,http://www.imdb.com/title/tt0449088/?ref_=fn_t...,"1,238.00",English,USA,PG-13,"300,000,000.00","2,007.00","5,000.00",7.10,2.35,0
2,Color,Sam Mendes,602.00,148.00,0.00,161.00,Rory Kinnear,"11,000.00","200,074,175.00",Action|Adventure|Thriller,Christoph Waltz,Spectre,275868,11700,Stephanie Sigman,1.00,bomb|espionage|sequel|spy|terrorist,http://www.imdb.com/title/tt2379713/?ref_=fn_t...,994.00,English,UK,PG-13,"245,000,000.00","2,015.00",393.00,6.80,2.35,85000
3,Color,Christopher Nolan,813.00,164.00,"22,000.00","23,000.00",Christian Bale,"27,000.00","448,130,642.00",Action|Thriller,Tom Hardy,The Dark Knight Rises,1144337,106759,Joseph Gordon-Levitt,0.00,deception|imprisonment|lawlessness|police offi...,http://www.imdb.com/title/tt1345836/?ref_=fn_t...,"2,701.00",English,USA,PG-13,"250,000,000.00","2,012.00","23,000.00",8.50,2.35,164000
5,Color,Andrew Stanton,462.00,132.00,475.00,530.00,Samantha Morton,640.00,"73,058,679.00",Action|Adventure|Sci-Fi,Daryl Sabara,John Carter,212204,1873,Polly Walker,1.00,alien|american civil war|male nipple|mars|prin...,http://www.imdb.com/title/tt0401729/?ref_=fn_t...,738.00,English,USA,PG-13,"263,700,000.00","2,012.00",632.00,6.60,2.35,24000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5026,Color,Olivier Assayas,81.00,110.00,107.00,45.00,Béatrice Dalle,576.00,"136,007.00",Drama|Music|Romance,Maggie Cheung,Clean,3924,776,Don McKellar,1.00,jail|junkie|money|motel|singer,http://www.imdb.com/title/tt0388838/?ref_=fn_t...,39.00,French,France,R,"4,500.00","2,004.00",133.00,6.90,2.35,171
5027,Color,Jafar Panahi,64.00,90.00,397.00,0.00,Nargess Mamizadeh,5.00,"673,780.00",Drama,Fereshteh Sadre Orafaiy,The Circle,4555,5,Mojgan Faramarzi,0.00,abortion|bus|hospital|prison|prostitution,http://www.imdb.com/title/tt0255094/?ref_=fn_t...,26.00,Persian,Iran,Not Rated,"10,000.00","2,000.00",0.00,7.50,1.85,697
5033,Color,Shane Carruth,143.00,77.00,291.00,8.00,David Sullivan,291.00,"424,760.00",Drama|Sci-Fi|Thriller,Shane Carruth,Primer,72639,368,Casey Gooden,0.00,changing the future|independent film|invention...,http://www.imdb.com/title/tt0390384/?ref_=fn_t...,371.00,English,USA,PG-13,"7,000.00","2,004.00",45.00,7.00,1.85,19000
5035,Color,Robert Rodriguez,56.00,81.00,0.00,6.00,Peter Marquardt,121.00,"2,040,920.00",Action|Crime|Drama|Romance|Thriller,Carlos Gallardo,El Mariachi,52055,147,Consuelo Gómez,0.00,assassin|death|guitar|gun|mariachi,http://www.imdb.com/title/tt0104815/?ref_=fn_t...,130.00,Spanish,USA,R,"7,000.00","1,992.00",20.00,6.90,1.37,0


In [39]:
df_movie_clean.shape

(3755, 28)

In [41]:
#verificar que no se tengan valores faltantes
df_movie_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3755 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      3755 non-null   object 
 1   director_name              3755 non-null   object 
 2   num_critic_for_reviews     3755 non-null   float64
 3   duration                   3755 non-null   float64
 4   director_facebook_likes    3755 non-null   float64
 5   actor_3_facebook_likes     3755 non-null   float64
 6   actor_2_name               3755 non-null   object 
 7   actor_1_facebook_likes     3755 non-null   float64
 8   gross                      3755 non-null   float64
 9   genres                     3755 non-null   object 
 10  actor_1_name               3755 non-null   object 
 11  movie_title                3755 non-null   object 
 12  num_voted_users            3755 non-null   int64  
 13  cast_total_facebook_likes  3755 non-null   int64  
 1

## tratar datos faltantes en columnas numéricas

In [42]:
# Para columnas numéricas
df_movie['duration']

0      178.00
1      169.00
2      148.00
3      164.00
4         NaN
        ...  
5038    87.00
5039    43.00
5040    76.00
5041   100.00
5042    90.00
Name: duration, Length: 5043, dtype: float64

In [43]:
#promedio
df_movie['duration'].fillna(df_movie['duration'].mean())
df_movie

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,actor_1_name,movie_title,num_voted_users,cast_total_facebook_likes,actor_3_name,facenumber_in_poster,plot_keywords,movie_imdb_link,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.00,178.00,0.00,855.00,Joel David Moore,"1,000.00","760,505,847.00",Action|Adventure|Fantasy|Sci-Fi,CCH Pounder,Avatar,886204,4834,Wes Studi,0.00,avatar|future|marine|native|paraplegic,http://www.imdb.com/title/tt0499549/?ref_=fn_t...,"3,054.00",English,USA,PG-13,"237,000,000.00","2,009.00",936.00,7.90,1.78,33000
1,Color,Gore Verbinski,302.00,169.00,563.00,"1,000.00",Orlando Bloom,"40,000.00","309,404,152.00",Action|Adventure|Fantasy,Johnny Depp,Pirates of the Caribbean: At World's End,471220,48350,Jack Davenport,0.00,goddess|marriage ceremony|marriage proposal|pi...,http://www.imdb.com/title/tt0449088/?ref_=fn_t...,"1,238.00",English,USA,PG-13,"300,000,000.00","2,007.00","5,000.00",7.10,2.35,0
2,Color,Sam Mendes,602.00,148.00,0.00,161.00,Rory Kinnear,"11,000.00","200,074,175.00",Action|Adventure|Thriller,Christoph Waltz,Spectre,275868,11700,Stephanie Sigman,1.00,bomb|espionage|sequel|spy|terrorist,http://www.imdb.com/title/tt2379713/?ref_=fn_t...,994.00,English,UK,PG-13,"245,000,000.00","2,015.00",393.00,6.80,2.35,85000
3,Color,Christopher Nolan,813.00,164.00,"22,000.00","23,000.00",Christian Bale,"27,000.00","448,130,642.00",Action|Thriller,Tom Hardy,The Dark Knight Rises,1144337,106759,Joseph Gordon-Levitt,0.00,deception|imprisonment|lawlessness|police offi...,http://www.imdb.com/title/tt1345836/?ref_=fn_t...,"2,701.00",English,USA,PG-13,"250,000,000.00","2,012.00","23,000.00",8.50,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.00,NaN,Rob Walker,131.00,NaN,Documentary,Doug Walker,Star Wars: Episode VII - The Force Awakens ...,8,143,NaN,0.00,NaN,http://www.imdb.com/title/tt5289954/?ref_=fn_t...,NaN,NaN,NaN,NaN,NaN,NaN,12.00,7.10,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5038,Color,Scott Smith,1.00,87.00,2.00,318.00,Daphne Zuniga,637.00,NaN,Comedy|Drama,Eric Mabius,Signed Sealed Delivered,629,2283,Crystal Lowe,2.00,fraud|postal worker|prison|theft|trial,http://www.imdb.com/title/tt3000844/?ref_=fn_t...,6.00,English,Canada,NaN,NaN,"2,013.00",470.00,7.70,NaN,84
5039,Color,NaN,43.00,43.00,NaN,319.00,Valorie Curry,841.00,NaN,Crime|Drama|Mystery|Thriller,Natalie Zea,The Following,73839,1753,Sam Underwood,1.00,cult|fbi|hideout|prison escape|serial killer,http://www.imdb.com/title/tt2071645/?ref_=fn_t...,359.00,English,USA,TV-14,NaN,NaN,593.00,7.50,16.00,32000
5040,Color,Benjamin Roberds,13.00,76.00,0.00,0.00,Maxwell Moody,0.00,NaN,Drama|Horror|Thriller,Eva Boehnke,A Plague So Pleasant,38,0,David Chandler,0.00,NaN,http://www.imdb.com/title/tt2107644/?ref_=fn_t...,3.00,English,USA,NaN,"1,400.00","2,013.00",0.00,6.30,NaN,16
5041,Color,Daniel Hsia,14.00,100.00,0.00,489.00,Daniel Henney,946.00,"10,443.00",Comedy|Drama|Romance,Alan Ruck,Shanghai Calling,1255,2386,Eliza Coupe,5.00,NaN,http://www.imdb.com/title/tt2070597/?ref_=fn_t...,9.00,English,USA,PG-13,NaN,"2,012.00",719.00,6.30,2.35,660


In [44]:
# Opción 2: imputar una columna numérica con la mediana.
# La mediana suele ser más resistente a valores extremos que el promedio.
df_movie['duration'].median()

np.float64(103.0)

## Estrategias iniciales para valores faltantes

La estrategia depende del tipo de variable y del contexto. Elimina filas solo si la pérdida es pequeña y no introduce sesgo; imputa con estadísticas calculadas en el conjunto de entrenamiento cuando prepares un modelo.

Para categorías, `Unknown` debe distinguirse de una categoría real. En un proyecto, documenta cuántos valores fueron imputados y por qué.

In [45]:
df_movie['color'].unique()

array(['Color', nan, ' Black and White'], dtype=object)

## Actividad de clase

Responde y justifica tus decisiones. No existe una única respuesta correcta si explicas el criterio.

1. Clasifica `job`, `education`, `month`, `campaign` y `y` según su significado estadístico.
2. ¿Por qué no sería correcto calcular el promedio de `job` aunque se codifique con números?
3. ¿Qué problema puede aparecer si se codifica `month` como `1, 2, ..., 12` y se usa esa columna directamente en un modelo?
4. Calcula la proporción de personas con `y == 'yes'` y compárala por nivel de `education`.
5. Elige entre eliminar o imputar los faltantes de `movie_metadata.csv`. Reporta cuántas filas o valores afecta tu decisión.

### Reto

Construye un diccionario llamado `taxonomia` con tres llaves: `binarias`, `nominales` y `ordinales`. Después verifica que ninguna columna categórica quede sin clasificar.

In [47]:
df_bank

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no
